# 数据加载、存储和文件格式
读取数据并使其可访问（通常称为数据加载）是使用本书中大多数工具的必要第一步。解析一词有时也用来描述加载文本数据并将其解释为表格和不同数据类型的过程。我将专注于使用pandas进行数据输入和输出，尽管其他库中有许多工具可以帮助读取和写入各种格式的数据。

输入和输出通常分为几个主要类别：读取文本文件和其他更高效的磁盘格式、从数据库加载数据以及与网络资源（如Web API）交互。

## 以文本格式读写数据

pandas提供了许多用于将表格数据读取为DataFrame对象的函数。下表总结了一些常用的函数；pandas.read_csv是本书中最常用的函数之一。我们将在稍后讨论二进制数据格式。

| 方法 | 描述 |
|------|-----|
| read_csv | 从文件、URL或类似文件的对象中加载分隔符数据；默认使用逗号作为分隔符 |
| read_fwf | 以固定宽度列格式读取数据（即无分隔符）|
| read_clipboard | read_csv的变体，用于从剪贴板读取数据；适用于转换网页中的表格 |
| read_excel | 从Excel XLS或XLSX文件中读取表格数据 |
| read_hdf | 读取pandas写入的HDF5文件 |
| read_html | 读取给定HTML文档中所有表格 |
| read_json | 从JSON（JavaScript对象表示法）字符串表示、文件、URL或类文件对象中读取数据 |
| read_feather | 读取 Feather 二进制文件格式 |
| read_orc | 阅读Apache ORC二进制文件格式 |
| read_parquet | 阅读Apache Parquet二进制文件格式 |
| read_pickle | 使用Python pickle格式读取pandas存储的对象 |
| read_sas | 读取存储在SAS系统自定义存储格式之一中的SAS数据集 |
| read_spss | 读取由SPSS创建的数据文件 |
| read_sql | 读取SQL查询的结果（使用SQLAlchemy） |
| read_sql_table | 读取整个SQL表（使用SQLAlchemy）；相当于使用一个查询来选择该表中的所有内容，使用read_sql |
| read_stata | 从Stata文件格式读取数据集 |
| read_xml | 从XML文件中读取数据表 |

我将概述这些函数的机制，它们旨在将文本数据转换为DataFrame。这些函数的可选参数可能分为几类：

**索引**

可以将一个或多个列视为返回的DataFrame，以及是否从文件、您提供的参数中获取列名，或者根本不获取。

**类型推断和数据转换**

包括用户定义的值转换和自定义的缺失值标记列表。

**日期和时间解析**

包括合并功能，可以将多个列中的日期和时间信息合并到结果中的一列中。

**迭代**

支持迭代遍历非常大的文件。

**脏数据问题**

包括跳行或页脚、注释或其他小事情，比如用逗号分隔的千位数。

由于现实世界中的数据可能非常杂乱，一些数据加载函数（特别是pandas.read_csv）随着时间的推移积累了许多可选参数。面对不同参数的数量感到不知所措是很正常的（pandas.read_csv大约有50个参数）。在线的pandas文档中有许多示例说明了每个参数是如何工作的，所以如果你在阅读某个特定文件时遇到困难，可能会有一个足够相似的示例来帮助你找到正确的参数。

这些功能中的一些执行类型推断，因为列数据类型不是数据格式的一部分。这意味着你不必指定哪些列是数值型、整数型、布尔型或字符串型。其他数据格式（如HDF5、ORC和Parquet）将数据类型信息嵌入到格式中。

处理日期和其他自定义类型可能需要额外的努力。

让我们从一个小的逗号分隔值（CSV）文本文件开始：

In [1]:
!cat examples/ex1.csv

a,b,c,d,message
1,2,3,4,hello
5,6,7,8,world
9,10,11,12,foo

> 在这里我使用了Unix的cat命令将文件的原始内容打印到屏幕上。如果你在Windows上，可以使用type代替cat来实现相同的效果，在Windows终端（或命令行）中。

由于这是以逗号分隔的，我们可以使用pandas.read_csv将其读取到DataFrame中：

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('examples/ex1.csv')
df

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


文件并不总是有标题行。考虑这个文件：

In [3]:
!cat examples/ex2.csv

1,2,3,4,hello
5,6,7,8,world
9,10,11,12,foo

要读取此文件，您有几个选项。您可以让pandas分配默认的列名，或者您可以自己指定名称：

In [4]:
pd.read_csv('examples/ex2.csv', header=None)

,0,1,2,3,4
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


In [5]:
pd.read_csv('examples/ex2.csv', names=["a", "b", "c", "d", "message"])

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


假设您希望message列是返回的DataFrame的索引。您可以使用index_col参数指定您希望第4列或名为“message”的列作为索引：

In [6]:
names = ["a", "b", "c", "d", "message"]
pd.read_csv('examples/ex2.csv', names=names, index_col='message')

,a,b,c,d
message,,,,
hello,1,2,3,4
world,5,6,7,8
foo,9,10,11,12


如果您想从多个列中形成层次索引（在第8.1章“层次索引”中讨论），请传递一个列号或名称列表：

In [7]:
!cat examples/csv_mindex.csv

key1,key2,value1,value2
one,a,1,2
one,b,3,4
one,c,5,6
one,d,7,8
two,a,9,10
two,b,11,12
two,c,13,14
two,d,15,16


In [8]:
parsed = pd.read_csv('examples/csv_mindex.csv', index_col=['key1', 'key2'])
parsed

value1  value2
key1 key2                
one  a          1       2
     b          3       4
     c          5       6
     d          7       8
two  a          9      10
     b         11      12
     c         13      14
     d         15      16

在某些情况下，表格可能没有固定的分隔符，而是使用空格或其他模式来分隔字段。考虑这样一个文本文件：

In [9]:
!cat examples/ex3.txt

            A         B         C
aaa -0.264438 -1.026059 -0.619500
bbb  0.927272  0.302904 -0.032399
ccc -0.264273 -0.386314 -0.217601
ddd -0.871858 -0.348382  1.100491


虽然你可以手动做一些处理，但这里的字段被不同数量的空格分隔。在这些情况下，你可以将正则表达式作为pandas.read_csv的分隔符传递。这可以通过正则表达式\s+来表达，因此我们有：

In [10]:
result = pd.read_csv("examples/ex3.txt", sep=r"\s+")
result

,A,B,C
aaa,-0.264438,-1.026059,-0.619500
bbb,0.927272,0.302904,-0.032399
ccc,-0.264273,-0.386314,-0.217601
ddd,-0.871858,-0.348382,1.100491


因为列名比数据行少一个，pandas.read_csv在这种情况下推断第一列应该是DataFrame的索引。

文件解析函数有许多额外的参数可以帮助您处理各种异常文件格式（见表 6.2 中的部分列表）。例如，您可以使用 skiprows 跳过文件的第一行、第三行和第四行：

In [11]:
!cat examples/ex4.csv

# hey!
a,b,c,d,message
# just wanted to make things more difficult for you
# who reads CSV files with computers, anyway?
1,2,3,4,hello
5,6,7,8,world
9,10,11,12,foo


In [12]:
pd.read_csv('examples/ex4.csv', skiprows=[0, 2, 3])

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


处理缺失值是文件读取过程中一个重要且经常需要细致考虑的部分。缺失数据通常要么不存在（空字符串），要么被某个哨兵（占位符）值标记。默认情况下，pandas使用一组常见的哨兵值，如NA和NULL：

In [13]:
!cat examples/ex5.csv

something,a,b,c,d,message
one,1,2,3,4,NA
two,5,6,,8,world
three,9,10,11,12,foo

In [14]:
result = pd.read_csv('examples/ex5.csv')
result

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,foo


请记住，pandas将缺失值输出为NaN，因此我们的结果中有两个空值或缺失值：

In [15]:
pd.isna(result)

,something,a,b,c,d,message
0,False,False,False,False,False,True
1,False,False,False,True,False,False
2,False,False,False,False,False,False


na_values选项接受一系列字符串，以添加到默认的缺失字符串列表中：

In [16]:
result = pd.read_csv('examples/ex5.csv', na_values=['NULL'])
result

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,foo


pandas.read_csv有很多默认的NA值表示方法，但是这些默认值可以通过keep_default_na选项禁用：

In [17]:
result2 = pd.read_csv('examples/ex5.csv', keep_default_na=False)
result2

,something,a,b,c,d,message
0,one,1,2,3,4,NA
1,two,5,6,,8,world
2,three,9,10,11,12,foo


In [18]:
result2.isna()

,something,a,b,c,d,message
0,False,False,False,False,False,False
1,False,False,False,False,False,False
2,False,False,False,False,False,False


In [19]:
result3 = pd.read_csv('examples/ex5.csv', keep_default_na=False, na_values=['NA'])

result3

,something,a,b,c,d,message
0,one,1,2,3,4,NaN
1,two,5,6,,8,world
2,three,9,10,11,12,foo


In [20]:
result3.isna()

,something,a,b,c,d,message
0,False,False,False,False,False,True
1,False,False,False,False,False,False
2,False,False,False,False,False,False


可以为字典中的每一列指定不同的NA哨兵：

In [21]:
sentinels = {"message": ["foo", "NA"], "something": ["two"]}

pd.read_csv('examples/ex5.csv', na_values=sentinels, keep_default_na=False)

,something,a,b,c,d,message
0,one,1,2,3,4,NaN
1,NaN,5,6,,8,world
2,three,9,10,11,12,NaN


**一些pandas.read_csv函数参数:**

| 参数 | 描述 |
|------|-----|
| path | 指示文件系统位置、URL或类似文件的字符串。 |
| sep / delimiter | 用于分割每行中字段的字符序列或正则表达式。 |
| header | 用作列名的行号；默认为0（第一行），但如果没有标题行则应为None。 |
| index_col | 要作为结果中行索引使用的列号或名称；可以是单个名称/数字，也可以是用于层次化索引的多个名称/数字列表。|
| names | 结果列名列表。|
| skiprows | 要忽略的文件开头的行数或要跳过的行号列表（从0开始）。|
| na_values | 要替换为NA的值序列。除非传递keep_default_na=False，否则它们将添加到默认列表中。|
| keep_default_na | 是否使用默认的NA值列表（默认为True）。|
| comment | 用于将注释从行尾分开的字符。|
| parse_dates | 尝试将数据解析为日期时间；默认为False。如果为True，将尝试解析所有列。否则，可以指定要解析的列号或列名列表。如果列表中的元素是元组或列表，将合并多个列并解析为日期（例如，如果日期/时间跨越两列）。|
| keep_date_col | 如果要将列连接起来解析日期，请保留连接的列；默认值为False。 |
| converters | 字典包含列号或名称映射到函数（例如，{"foo": f} 会将函数f应用于“foo”列中的所有值）。 |
| dayfirst | 在解析可能产生歧义的日期时，将其视为国际格式（例如，7/6/2012 -> 2012年6月7日）；默认值为假。|
| date_parser | 用于解析日期的函数。|
| nrows | 从文件开头读取的行数（不包括标题）。|
| iterator | 返回一个TextFileReader对象，用于分块读取文件。这个对象也可以与with语句一起使用。|
| chunksize | 用于迭代，文件块的大小。|
| skip_footer | 在文件末尾忽略的行数。|
| verbose | 打印各种解析信息，如文件转换中每个阶段所花费的时间以及内存使用情况。|
| encoding | 文本编码（例如，“utf-8”用于UTF-8编码的文本）。如果未指定，则默认为“utf-8”。|
| squeeze | 如果解析的数据只有一列，则返回一个Series。|
| thousands | 千位分隔符（例如逗号或分号）。默认值为None。|
| decimal | 数字中的小数分隔符（例如"."或","）；默认是"." |
| engine | 要使用的CSV解析和转换引擎；可以是“c”、“python”或“pyarrow”。默认是“c”，尽管较新的“pyarrow”引擎可以更快地解析某些文件。 “python”引擎速度较慢，但支持其他引擎不支持的一些功能。|

### 分块读取文本文件

在处理非常大的文件或确定正确的一组参数来正确处理大型文件时，您可能只想读取文件的一小部分或迭代通过文件的较小部分。

在我们查看大文件之前，我们让pandas显示设置更紧凑：

In [22]:
pd.options.display.max_rows = 10

In [23]:
result = pd.read_csv('examples/ex6.csv')

result

,one,two,three,four,key
0,0.467976,-0.038649,-0.295344,-1.824726,L
1,-0.358893,1.404453,0.704965,-0.200638,B
2,-0.501840,0.659254,-0.421691,-0.057688,G
3,0.204886,1.074134,1.388361,-0.982404,R
4,0.354628,-0.133116,0.283763,-0.837063,Q
...,...,...,...,...,...
9995,2.311896,-0.417070,-1.409599,-0.515821,L
9996,-0.479893,-0.650419,0.745152,-0.646038,E
9997,0.523331,0.787112,0.486066,1.093156,K
9998,-0.362559,0.598894,-1.843201,0.887292,G


省略号标记`...`表示数据框中间的行已被省略。

如果您只想读取少量行（避免读取整个文件），请使用nrows指定：

In [24]:
pd.read_csv('examples/ex6.csv', nrows=5)

,one,two,three,four,key
0,0.467976,-0.038649,-0.295344,-1.824726,L
1,-0.358893,1.404453,0.704965,-0.200638,B
2,-0.501840,0.659254,-0.421691,-0.057688,G
3,0.204886,1.074134,1.388361,-0.982404,R
4,0.354628,-0.133116,0.283763,-0.837063,Q


要分块读取文件，请指定一个以行为单位的chunksize：

In [25]:
chunker = pd.read_csv('examples/ex6.csv', chunksize=1000)
chunker

pandas.read_csv返回的TextFileReader对象允许您根据chunksize迭代文件的各个部分。例如，我们可以遍历ex6.csv，聚合“key”列中的值计数，如下所示：

In [26]:
chunker = pd.read_csv('examples/ex6.csv', chunksize=1000)
tot = pd.Series([], dtype="int64")
for piece in chunker:
    tot = tot.add(piece['key'].value_counts(), fill_value=0)
tot = tot.sort_values(ascending=False)
tot

key
E    368.0
X    364.0
L    346.0
O    343.0
Q    340.0
     ...  
5    157.0
2    152.0
0    151.0
9    150.0
1    146.0
Length: 36, dtype: float64

TextFileReader还配备了一个get_chunk方法，它允许您读取任意大小的片段。

### 将数据写入文本格式

数据也可以导出到分隔格式。让我们考虑之前读取的其中一个CSV文件：

In [27]:
data = pd.read_csv('examples/ex5.csv')
data

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,foo


使用DataFrame的to_csv方法，我们可以将数据写入一个逗号分隔的文件：

In [28]:
data.to_csv('examples/out.csv')
!cat examples/out.csv

,something,a,b,c,d,message
0,one,1,2,3.0,4,
1,two,5,6,,8,world
2,three,9,10,11.0,12,foo


当然也可以使用其他分隔符（将文本写入sys.stdout，以便在控制台上打印文本结果而不是文件）：

In [29]:
import sys
data.to_csv(sys.stdout, sep='|')

|something|a|b|c|d|message
0|one|1|2|3.0|4|
1|two|5|6||8|world
2|three|9|10|11.0|12|foo


缺失值在输出中显示为空字符串。您可能希望用其他哨兵值来表示它们：

In [30]:
data.to_csv(sys.stdout, na_rep='NULL')

,something,a,b,c,d,message
0,one,1,2,3.0,4,NULL
1,two,5,6,NULL,8,world
2,three,9,10,11.0,12,foo


如果没有指定其他选项，将同时写入行和列标签。这两者都可以禁用：

In [31]:
data.to_csv(sys.stdout, index=False, header=False)

one,1,2,3.0,4,
two,5,6,,8,world
three,9,10,11.0,12,foo


您也可以只写部分列，并且按您选择的顺序写：

In [32]:
data.to_csv(sys.stdout, index=False, columns=['a', 'b', 'c'])

a,b,c
1,2,3.0
5,6,
9,10,11.0


### 与其他分隔格式一起工作

可以使用pandas.read_csv等函数从磁盘加载大多数表格数据。然而，在某些情况下，可能需要一些手动处理。收到一个或多个格式错误的行可能会导致pandas.read_csv出错的情况并不罕见。为了说明基本工具，考虑一个小型CSV文件：

In [33]:
!cat examples/ex7.csv

"a","b","c"
"1","2","3"
"1","2","3"


对于任何以单字符分隔符分隔的文件，你可以使用Python内置的csv模块。要使用它，将任何打开的文件或类似文件的对象传递给csv.reader：

In [34]:
import csv

f = open('examples/ex7.csv')
reader = csv.reader(f)

像处理文件一样迭代读取器会生成一个列表，其中所有引号字符都被移除：

In [35]:
for line in reader:
    print(line)

f.close()

['a', 'b', 'c']
['1', '2', '3']
['1', '2', '3']


从这里开始，你需要进行必要的整理，将数据转换成你需要的格式。让我们一步一步来。首先，我们将文件读入一个行列表中：

In [36]:
with open('examples/ex7.csv') as f:
    lines = list(csv.reader(f))

然后我们将行分为标题行和数据行：

In [37]:
header, values = lines[0], lines[1:]

然后我们可以使用字典推导式和表达式zip(*values)（注意这会在大型文件上占用大量内存）创建一个数据列的字典，该表达式将行转置成列：

In [38]:
data_dict = {h: v for h, v in zip(header, zip(*values))}
data_dict

{'a': ('1', '1'), 'b': ('2', '2'), 'c': ('3', '3')}

CSV文件有许多不同的格式。要定义一种具有不同分隔符、字符串引用约定或行终止符的新格式，我们可以定义csv.Dialect的一个简单子类：

In [39]:
class my_dialect(csv.Dialect):
    lineterminator = '\n'
    delimiter = ';'
    quotechar = '"'
    quoting = csv.QUOTE_MINIMAL

In [40]:
f = open("examples/ex7.csv")
reader = csv.reader(f, dialect=my_dialect)

我们也可以将单个CSV方言参数作为关键词传递给csv.reader，而不必定义一个子类：

In [41]:
reader = csv.reader(f, delimiter="|")

**CSV方言选项:**

| 参数 | 描述 |
|------|-----|
| delimiter | 用于分隔字段的单字符字符串；默认值为“,”。|
| lineterminator | 用于写入的换行符；默认值为“\r\n”。读取器忽略此值并识别跨平台的换行符。|
| quotechar | 用于包含特殊字符（如分隔符）字段的引号字符；默认值为双引号。|
| quoting | 引用约定。选项包括csv.QUOTE_ALL（引用所有字段）、csv.QUOTE_MINIMAL（只引用包含特殊字符如分隔符的字段）、csv.QUOTE_NONNUMERIC和csv.QUOTE_NONE（无引用）。详见Python文档以获取详细信息。默认值为QUOTE_MINIMAL。|
| skipinitialspace | 忽略每个分隔符后面的空格；默认值为False。|
| doublequote | 如何处理字段内的引号字符；如果为True，则将其加倍（请参阅在线文档以获取完整细节和行为）。|
| escapechar | 如果将引用设置为csv.QUOTE_NONE，则要转义的字符串；默认情况下是禁用的。|


> 对于具有更复杂或固定多字符分隔符的文件，您将无法使用csv模块。在这些情况下，您必须使用字符串的split方法或正则表达式方法re.split进行行分割和其他清理工作。幸运的是，如果传递了必要的选项，pandas.read_csv几乎可以做你需要的一切，所以你很少需要手动解析文件。

要手动编写分隔文件，可以使用csv.writer。它接受一个打开的、可写的文件对象以及和csv.reader相同的方言和格式选项：

In [42]:
with open('mydata.csv', 'w') as f:
    writer = csv.writer(f, dialect=my_dialect)
    writer.writerow(("one", "two", "three"))
    writer.writerow(("1", "2", "3"))
    writer.writerow(("4", "5", "6"))
    writer.writerow(("7", "8", "9"))

### JSON数据

JSON（JavaScript对象表示法）已成为通过HTTP请求在Web浏览器和其他应用程序之间发送数据的标准格式之一。它是一种比CSV等表格文本形式更加自由格式的数据格式。这里有一个例子：

In [43]:
obj = """
{"name": "Wes",
 "cities_lived": ["Akron", "Nashville", "New York", "San Francisco"],
 "pet": null,
 "siblings": [{"name": "Scott", "age": 34, "hobbies": ["guitars", "soccer"]},
              {"name": "Katie", "age": 42, "hobbies": ["diving", "art"]}]
}
"""

JSON几乎是有效的Python代码，除了它的空值null和一些其他细微差别（例如不允许列表末尾有尾随逗号）。基本类型包括对象（字典）、数组（列表）、字符串、数字、布尔值和空值。对象中的所有键都必须是字符串。有几个Python库用于读取和写入JSON数据。我将使用json，因为它内置于Python标准库中。要将JSON字符串转换为Python形式，请使用json.loads：

In [44]:
import json

result = json.loads(obj)

result

{'name': 'Wes',
 'cities_lived': ['Akron', 'Nashville', 'New York', 'San Francisco'],
 'pet': None,
 'siblings': [{'name': 'Scott', 'age': 34, 'hobbies': ['guitars', 'soccer']},
  {'name': 'Katie', 'age': 42, 'hobbies': ['diving', 'art']}]}

另一方面，json.dumps将Python对象转换回JSON：

In [45]:
asjson = json.dumps(result)

asjson

'{"name": "Wes", "cities_lived": ["Akron", "Nashville", "New York", "San Francisco"], "pet": null, "siblings": [{"name": "Scott", "age": 34, "hobbies": ["guitars", "soccer"]}, {"name": "Katie", "age": 42, "hobbies": ["diving", "art"]}]}'

如何将JSON对象或对象列表转换为DataFrame或其他数据结构进行分析将由您决定。方便的是，您可以将字典列表（之前是JSON对象）传递给DataFrame构造函数并选择数据字段的子集：

In [46]:
siblings = pd.DataFrame(result['siblings'], columns=['name', 'age'])

siblings

,name,age
0,Scott,34
1,Katie,42


pandas.read_json可以自动将特定排列的JSON数据集转换为Series或DataFrame。例如：

In [47]:
!cat examples/example.json

[{"a": 1, "b": 2, "c": 3},
 {"a": 4, "b": 5, "c": 6},
 {"a": 7, "b": 8, "c": 9}]


pandas.read_json的默认选项假设JSON数组中的每个对象都是表中的一行：

In [48]:
data = pd.read_json('examples/example.json')

data

,a,b,c
0,1,2,3
1,4,5,6
2,7,8,9


有关读取和操作JSON数据的扩展示例（包括嵌套记录），请参见第13章：数据分析示例中的美国农业部食品数据库示例。

如果你需要将pandas数据导出到JSON格式，一种方法是使用Series和DataFrame的to_json方法：

In [49]:
data.to_json(sys.stdout)

{"a":{"0":1,"1":4,"2":7},"b":{"0":2,"1":5,"2":8},"c":{"0":3,"1":6,"2":9}}

In [50]:
data.to_json(sys.stdout, orient="records")

[{"a":1,"b":2,"c":3},{"a":4,"b":5,"c":6},{"a":7,"b":8,"c":9}]

### XML和HTML：网络爬虫

Python有许多库用于读取和写入普遍存在的HTML和XML格式的数据。例如，lxml、Beautiful Soup和html5lib。虽然lxml通常在速度上相对较快，但其他库可以更好地处理格式错误的HTML或XML文件。

pandas有一个内置函数，pandas.read_html，它使用所有这些库来自动解析HTML文件中的表格，并将其作为DataFrame对象。为了展示这是如何工作的，我从美国联邦存款保险公司（US FDIC）下载了一个HTML文件（用于pandas文档），该文件显示了银行失败的情况。首先，您必须安装read_html使用的某些附加库：
`conda install lxml beautifulsoup4 html5lib`

如果你不使用conda，`pip install lxml`也应该可以。

pandas.read_html函数有许多选项，但默认情况下它会搜索并尝试解析所有包含在<table>标签内的表格数据。结果是DataFrame对象的列表：

In [51]:
tables = pd.read_html("examples/fdic_failed_bank_list.html")
len(tables)

1

In [52]:
failures = tables[0]
failures.head()

,Bank Name,City,ST,CERT,Acquiring Institution,Closing Date,Updated Date
0,Allied Bank,Mulberry,AR,91,Today's Bank,"September 23, 2016","November 17, 2016"
1,The Woodbury Banking Company,Woodbury,GA,11297,United Bank,"August 19, 2016","November 17, 2016"
2,First CornerStone Bank,King of Prussia,PA,35312,First-Citizens Bank & Trust Company,"May 6, 2016","September 6, 2016"
3,Trust Company Bank,Memphis,TN,9956,The Bank of Fayette County,"April 29, 2016","September 6, 2016"
4,North Milwaukee State Bank,Milwaukee,WI,20364,First-Citizens Bank & Trust Company,"March 11, 2016","June 16, 2016"


正如你在后面的章节中将会学到的，从这里我们可以继续做一些数据清洗和分析工作，比如计算每年的银行倒闭数量：

In [53]:
close_timestamps = pd.to_datetime(failures['Closing Date'])

close_timestamps.dt.year.value_counts()

Closing Date
2010    157
2009    140
2011     92
2012     51
2008     25
       ... 
2004      4
2001      4
2007      3
2003      3
2000      2
Name: count, Length: 15, dtype: int64

#### 使用lxml.objectify解析XML

XML是另一种常见的结构化数据格式，它支持具有元数据的层次化、嵌套数据。你目前正在阅读的书实际上是从一系列大型XML文档中创建的。

之前我展示了pandas.read_html函数，该函数在幕后使用lxml或Beautiful Soup来解析HTML中的数据。XML和HTML在结构上相似，但XML更为通用。在这里，我将展示如何使用lxml来解析来自更通用XML格式的数据。

多年来，纽约大都会交通管理局（MTA）发布了一系列关于其公交和火车服务的XML格式数据系列。在这里，我们将查看包含在一组XML文件中的性能数据。每个火车或公交服务都有一个不同的文件（例如，地铁-北部铁路的Performance_MNR.xml），其中包含每月数据作为一系列XML记录，如下所示：

```xml
<INDICATOR>
  <INDICATOR_SEQ>373889</INDICATOR_SEQ>
  <PARENT_SEQ></PARENT_SEQ>
  <AGENCY_NAME>Metro-North Railroad</AGENCY_NAME>
  <INDICATOR_NAME>Escalator Availability</INDICATOR_NAME>
  <DESCRIPTION>Percent of the time that escalators are operational
  systemwide. The availability rate is based on physical observations performed
  the morning of regular business days only. This is a new indicator the agency
  began reporting in 2009.</DESCRIPTION>
  <PERIOD_YEAR>2011</PERIOD_YEAR>
  <PERIOD_MONTH>12</PERIOD_MONTH>
  <CATEGORY>Service Indicators</CATEGORY>
  <FREQUENCY>M</FREQUENCY>
  <DESIRED_CHANGE>U</DESIRED_CHANGE>
  <INDICATOR_UNIT>%</INDICATOR_UNIT>
  <DECIMAL_PLACES>1</DECIMAL_PLACES>
  <YTD_TARGET>97.00</YTD_TARGET>
  <YTD_ACTUAL></YTD_ACTUAL>
  <MONTHLY_TARGET>97.00</MONTHLY_TARGET>
  <MONTHLY_ACTUAL></MONTHLY_ACTUAL>
</INDICATOR>
```
使用lxml.objectify解析文件，并使用getroot获取XML文件的根节点引用：

In [54]:
from lxml import objectify

path = "datasets/mta_perf/Performance_MNR.xml"

with open(path) as f:
    parsed = objectify.parse(f)

root = parsed.getroot()

root.INDICATOR 返回一个生成器，用于产生每个<INDICATOR> XML元素。对于每条记录，我们可以通过运行以下代码来填充一个标签名（如YTD_ACTUAL）到数据值（排除一些标签）的字典：

In [55]:
data = []

skip_fields = ["PARENT_SEQ", "INDICATOR_SEQ", 
               "DESIRED_CHANGE", "DECIMAL_PLACES"]

for elt in root.INDICATOR:
    el_data = {}
    for child in elt.getchildren():
        if child.tag in skip_fields:
            continue
        el_data[child.tag] = child.pyval
    data.append(el_data)

最后，将这个字典列表转换为DataFrame：

In [56]:
perf = pd.DataFrame(data)
perf.head()

,AGENCY_NAME,INDICATOR_NAME,DESCRIPTION,PERIOD_YEAR,PERIOD_MONTH,CATEGORY,FREQUENCY,INDICATOR_UNIT,YTD_TARGET,YTD_ACTUAL,MONTHLY_TARGET,MONTHLY_ACTUAL
0,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,1,Service Indicators,M,%,95.0,96.9,95.0,96.9
1,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,2,Service Indicators,M,%,95.0,96.0,95.0,95.0
2,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,3,Service Indicators,M,%,95.0,96.3,95.0,96.9
3,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,4,Service Indicators,M,%,95.0,96.8,95.0,98.3
4,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,5,Service Indicators,M,%,95.0,96.6,95.0,95.8


pandas的pandas.read_xml函数将这个过程简化为一行表达式：

In [57]:
perf2 = pd.read_xml(path)

perf2.head()

,INDICATOR_SEQ,PARENT_SEQ,AGENCY_NAME,INDICATOR_NAME,DESCRIPTION,PERIOD_YEAR,PERIOD_MONTH,CATEGORY,FREQUENCY,DESIRED_CHANGE,INDICATOR_UNIT,DECIMAL_PLACES,YTD_TARGET,YTD_ACTUAL,MONTHLY_TARGET,MONTHLY_ACTUAL
0,28445,NaN,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,1,Service Indicators,M,U,%,1,95.00,96.90,95.00,96.90
1,28445,NaN,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,2,Service Indicators,M,U,%,1,95.00,96.00,95.00,95.00
2,28445,NaN,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,3,Service Indicators,M,U,%,1,95.00,96.30,95.00,96.90
3,28445,NaN,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,4,Service Indicators,M,U,%,1,95.00,96.80,95.00,98.30
4,28445,NaN,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,5,Service Indicators,M,U,%,1,95.00,96.60,95.00,95.80


对于更复杂的XML文档，请参阅pandas.read_xml的文档字符串，了解如何执行选择和过滤以提取特定感兴趣的表格。

## 二进制数据格式

以二进制格式存储（或序列化）数据的一个简单方法是使用Python的内置pickle模块。pandas对象都有一个to_pickle方法，可以将数据以pickle格式写入磁盘：

In [58]:
frame = pd.read_csv('examples/ex1.csv')
frame

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


In [59]:
frame.to_pickle('examples/frame_pickle')

通常情况下，pickle文件只能在Python中读取。你可以直接使用内置的pickle来读取存储在文件中的任何“已序列化”对象，或者更方便地使用pandas.read_pickle：

In [60]:
pd.read_pickle('examples/frame_pickle')

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


> 建议仅将pickle作为短期存储格式。问题是很难保证该格式随时间保持稳定；今天pickle的对象可能无法用稍后版本的库反序列化。pandas在可能的情况下尝试保持向后兼容性，但在未来的某个时刻可能需要“打破”pickle格式。

pandas内置支持其他几种开源二进制数据格式，例如HDF5、ORC和Apache Parquet。例如，如果您安装了pyarrow包（conda install pyarrow），那么您可以使用pandas.read_parquet读取Parquet文件：

In [61]:
%pip install pyarrow
fec = pd.read_parquet('datasets/fec/fec.parquet')
fec.head()

Note: you may need to restart the kernel to use updated packages.


,cmte_id,cand_id,cand_nm,contbr_nm,contbr_city,contbr_st,contbr_zip,contbr_employer,contbr_occupation,contb_receipt_amt,contb_receipt_dt,receipt_desc,memo_cd,memo_text,form_tp,file_num
0,C00410118,P20002978,"Bachmann, Michelle","HARVEY, WILLIAM",MOBILE,AL,366010290,RETIRED,RETIRED,250.0,20-JUN-11,None,None,None,SA17A,736166
1,C00410118,P20002978,"Bachmann, Michelle","HARVEY, WILLIAM",MOBILE,AL,366010290,RETIRED,RETIRED,50.0,23-JUN-11,None,None,None,SA17A,736166
2,C00410118,P20002978,"Bachmann, Michelle","SMITH, LANIER",LANETT,AL,368633403,INFORMATION REQUESTED,INFORMATION REQUESTED,250.0,05-JUL-11,None,None,None,SA17A,749073
3,C00410118,P20002978,"Bachmann, Michelle","BLEVINS, DARONDA",PIGGOTT,AR,724548253,NONE,RETIRED,250.0,01-AUG-11,None,None,None,SA17A,749073
4,C00410118,P20002978,"Bachmann, Michelle","WARDENBURG, HAROLD",HOT SPRINGS NATION,AR,719016467,NONE,RETIRED,300.0,20-JUN-11,None,None,None,SA17A,736166


### 阅读Microsoft Excel文件

pandas还支持使用pandas.ExcelFile类或pandas.read_excel函数读取存储在Excel 2003（及更高版本）文件中的表格数据。这些工具内部使用了附加包xlrd和openpyxl来分别读取旧式XLS和新式XLSX文件。这些必须通过pip或conda与pandas分开安装：

`conda install openpyxl xlrd`

要使用pandas.ExcelFile，通过传递一个xls或xlsx文件的路径来创建一个实例：

In [62]:
%pip install openpyxl
xlsx = pd.ExcelFile('examples/ex1.xlsx')

Note: you may need to restart the kernel to use updated packages.


此对象可以显示文件中可用的工作表名称列表：

In [63]:
xlsx.sheet_names

['Sheet1']

然后可以使用parse将存储在表中的数据读入DataFrame：

In [64]:
xlsx.parse(sheet_name="Sheet1")

,Unnamed: 0,a,b,c,d,message
0,0,1,2,3,4,hello
1,1,5,6,7,8,world
2,2,9,10,11,12,foo


这个Excel表格有一个索引列，所以我们可以用index_col参数来指示它：

In [65]:
xlsx.parse(sheet_name="Sheet1", index_col=0)

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


如果你在文件中读取多张表，那么创建pandas.ExcelFile会更快，但你也可以直接将文件名传递给pandas.read_excel：

In [66]:
frame = pd.read_excel('examples/ex1.xlsx', 'Sheet1')
frame

,Unnamed: 0,a,b,c,d,message
0,0,1,2,3,4,hello
1,1,5,6,7,8,world
2,2,9,10,11,12,foo


要将pandas数据写入Excel格式，您必须首先创建一个ExcelWriter对象，然后使用pandas对象的to_excel方法将其数据写入其中：

In [67]:
writer = pd.ExcelWriter('examples/ex2.xlsx')
frame.to_excel(writer, sheet_name='Sheet1')
writer.close()

你也可以将文件路径传递给to_excel并避免使用ExcelWriter:

In [68]:
frame.to_excel('examples/ex2.xlsx', sheet_name='Sheet1')

### 使用HDF5格式

HDF5是一个备受推崇的文件格式，旨在存储大量的科学阵列数据。它作为一个C库提供，并且有许多其他语言（包括Java、Julia、MATLAB和Python）的接口可用。"HDF"在HDF5中代表层次化数据格式。每个HDF5文件可以存储多个数据集和支持元数据。与更简单的格式相比，HDF5支持多种压缩模式的即时压缩，使得具有重复模式的数据能够更有效地存储。对于不适合内存的数据集，HDF5是一个很好的选择，因为你可以高效地读取和写入更大数组的小部分。

要开始使用HDF5和pandas，您必须首先通过安装conda中的tables包来安装PyTables：

In [69]:
%pip install tables

Note: you may need to restart the kernel to use updated packages.


> 请注意，在PyPI中，PyTables包被称为“tables”，因此如果您使用pip安装，您必须运行pip install tables。

虽然可以使用PyTables或h5py库直接访问HDF5文件，但pandas提供了一个高级接口，简化了存储Series和DataFrame对象的过程。HDFStore类的工作方式类似于字典，并处理底层细节：

In [70]:
frame = pd.DataFrame({'a': np.random.randn(100)})

store = pd.HDFStore('examples/mydata.h5')
store['obj1'] = frame
store['obj1_col'] = frame['a']
store

<class 'pandas.io.pytables.HDFStore'>
File path: examples/mydata.h5

然后可以使用类似字典的API检索HDF5文件中包含的对象：

In [71]:
store['obj1']

,a
0,0.037524
1,0.718848
2,0.867459
3,-0.569876
4,-0.608255
...,...
95,1.480666
96,0.508435
97,0.625845
98,0.096297


HDFStore支持两种存储模式，“固定”和“表”（默认是“固定”）。后者通常较慢，但它支持使用特殊语法进行查询操作：

In [72]:
store.put('obj2', frame, format='table')
store.select('obj2', where=['index >= 10 and index <= 15'])

store.close()

put是store["obj2"] = frame方法的一个显式版本，但它允许我们设置其他选项，如存储格式。

pandas.read_hdf函数为你提供了这些工具的快捷方式：

In [73]:
frame.to_hdf('examples/mydata.h5', 'obj3', format='table')

/var/folders/c5/vh03t8zn4797kc18lgrjtcbr0000gn/T/ipykernel_15544/4067804331.py:1: FutureWarning: Starting with pandas version 3.0 all arguments of to_hdf except for the argument 'path_or_buf' will be keyword-only.
  frame.to_hdf('examples/mydata.h5', 'obj3', format='table')


In [74]:
pd.read_hdf("examples/mydata.h5", "obj3", where=["index < 5"])

,a
0,0.037524
1,0.718848
2,0.867459
3,-0.569876
4,-0.608255


> 如果您正在处理存储在远程服务器上的数据，例如Amazon S3或HDFS，使用专为分布式存储设计的不同二进制格式（如Apache Parquet）可能更为合适。

如果你在本地处理大量数据，我建议你探索一下PyTables和h5py，看看它们如何满足你的需求。由于许多数据分析问题都是I/O瓶颈（而不是CPU瓶颈），使用像HDF5这样的工具可以极大地加速你的应用程序。

> HDF5 不是数据库。它最适合于一次性写入、多次读取的数据集。虽然可以随时向文件中添加数据，但如果多个写者同时进行操作，文件可能会损坏。

## 与Web API交互

许多网站都有公共API提供数据源，通过JSON或其他格式。从Python访问这些API有多种方法；我推荐的一种方法是requests包，可以通过pip或conda安装：

In [75]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.


要在GitHub上找到pandas的最后30个GitHub问题，我们可以使用插件requests库进行GET HTTP请求：

In [76]:
import requests
url = "https://api.github.com/repos/pandas-dev/pandas/issues"
resp = requests.get(url)
resp.raise_for_status()

resp

<Response [200]>

在使用requests.get之后调用raise_for_status是一个好习惯，用于检查HTTP错误。

响应对象的json方法将返回一个Python对象，其中包含解析的JSON数据作为字典或列表（取决于返回的JSON类型）：

In [77]:
data = resp.json()
data[0]['title']

'BUG: Fix MultiIndex key type check for datetime/date (GH#55969)'

由于检索到的结果是基于实时数据的，因此当你运行此代码时，你看到的内容几乎肯定会不同。

data中的每个元素都是一个字典，包含了GitHub问题页面上找到的所有数据（评论除外）。我们可以直接将数据传递给pandas.DataFrame并提取感兴趣的字段：

In [78]:
issues = pd.DataFrame(data, columns=['number', 'title', 'labels', 'state'])
issues.head()

,number,title,labels,state
0,62819,BUG: Fix MultiIndex key type check for datetim...,[],open
1,62818,FIX: unstack(sort=False) data misalignment (#6...,[],open
2,62817,fix(#61434): Improve error message when mergin...,[],open
3,62816,BUG: `.unstack(sort=False)` reorders columns l...,"[{'id': 76811, 'node_id': 'MDU6TGFiZWw3NjgxMQ=...",open
4,62815,BUG: Correctly handle fill_value in Series._fl...,[],open


通过一些技巧，你可以创建一些高级接口来访问常见的Web API，这些API返回DataFrame对象以便更方便地进行分析。

## 与数据库交互

在商业环境中，许多数据可能不会存储在文本或Excel文件中。基于SQL的关系型数据库（如SQL Server、PostgreSQL和MySQL）被广泛使用，许多其他类型的数据库也变得相当流行。选择哪种数据库通常取决于应用程序的性能、数据完整性和可扩展性需求。

pandas有一些函数可以简化将SQL查询结果加载到DataFrame中的过程。例如，我将使用Python内置的sqlite3驱动程序创建一个SQLite3数据库：

In [84]:
import sqlite3
import os

os.remove('examples/test.db')
query = """CREATE TABLE test
(a VARCHAR(20), b VARCHAR(20),
c REAL, d INTEGER);"""
con = sqlite3.connect('examples/test.db')
con.execute(query)
con.commit()

然后，插入几行数据：

In [85]:
data = [("Atlanta", "Georgia", 1.25, 6),
        ("Tallahassee", "Florida", 2.6, 3),
        ("Sacramento", "California", 1.7, 5)]
stmt = "INSERT INTO test VALUES(?, ?, ?, ?)"
con.executemany(stmt, data)
con.commit()

大多数Python SQL驱动程序在从表中检索数据时返回一个元组列表：

In [86]:
cursor = con.execute('SELECT * FROM test')
rows = cursor.fetchall()
rows

[('Atlanta', 'Georgia', 1.25, 6),
 ('Tallahassee', 'Florida', 2.6, 3),
 ('Sacramento', 'California', 1.7, 5)]

你可以将元组列表传递给DataFrame构造函数，但还需要包含在游标描述属性中的列名。请注意，对于SQLite3，游标描述只提供列名（其他字段是Python的数据库API规范的一部分，它们是None），但对于某些其他数据库驱动程序，提供了更多的列信息：

In [87]:
cursor.description

(('a', None, None, None, None, None, None),
 ('b', None, None, None, None, None, None),
 ('c', None, None, None, None, None, None),
 ('d', None, None, None, None, None, None))

In [89]:
pd.DataFrame(rows, columns=[x[0] for x in cursor.description])

,a,b,c,d
0,Atlanta,Georgia,1.25,6
1,Tallahassee,Florida,2.60,3
2,Sacramento,California,1.70,5


这是相当多的混淆，你最好在每次查询数据库时都不要重复。SQLAlchemy项目是一个流行的Python SQL工具包，它抽象了许多SQL数据库之间的常见差异。pandas有一个read_sql函数，可以让你轻松地从一般的SQLAlchemy连接中读取数据。你可以像这样使用conda安装SQLAlchemy：

In [90]:
%pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.


现在，我们将使用SQLAlchemy连接到同一个SQLite数据库，并从之前创建的表中读取数据：

In [91]:
import sqlalchemy as sqla

db = sqla.create_engine('sqlite:///examples/test.db')
pd.read_sql('SELECT * FROM test', db)

,a,b,c,d
0,Atlanta,Georgia,1.25,6
1,Tallahassee,Florida,2.60,3
2,Sacramento,California,1.70,5
